In [ ]:
import torch
import torch.nn as nn

In [ ]:
# 1. Ensure dimensions are multiples of 8 for Tensor Core alignment
batch_size = 64  # Good
in_chars = 256   # Good
out_chars = 512  # Good

model = nn.Linear(in_chars, out_chars).cuda()
optimizer = torch.optim.Adam(model.parameters())

# 2. Initialize the GradScaler
# FP16 has low dynamic range; the scaler prevents "Gradient Underflow"
scaler = torch.amp.GradScaler('cuda')

In [ ]:
for inputs, targets in data_loader:
    inputs, targets = inputs.cuda(), targets.cuda()
    optimizer.zero_grad()

    # 3. Autocast context manager
    # This runs the forward pass in FP16 where beneficial
    with torch.amp.autocast('cuda'):
        output = model(inputs)
        loss = nn.functional.mse_loss(output, targets)

    # 4. Scales loss, calls backward, and unscales gradients for the optimizer
    scaler.scale(loss).backward()

    # scaler.step() first unscales the gradients.
    # If they contain NaNs/Infs, the step is skipped to avoid crashing training.
    scaler.step(optimizer)

    # Updates the scale factor for the next iteration
    scaler.update()